# D03 — Functional Genomics Datasets

Downloads IMPC mouse knockout phenotype data via the IMPC Solr API and
documents the Replogle et al. perturb-seq supplementary tables.

**Outputs (in `data/external/`):**
- `impc_phenotype_counts.csv` — IMPC phenotype counts per gene
- `1-s2/TabA_K562_day8_summary_stat-Table 1.csv` — Replogle K562 day-8 perturbation summary
- `1-s2/TabB_K562_day6_summary_stat-Table 1.csv` — Replogle K562 day-6 perturbation summary

**Prerequisites:** `requests`, `pandas`
**Note:** Replogle data must be manually downloaded from the publication supplementary materials.

## Section 1: IMPC Mouse Knockout Phenotypes

The International Mouse Phenotyping Consortium (IMPC) systematically knocks out every protein-coding gene in mice and measures phenotypic consequences across standardised assays. Genes with more significant phenotypes after knockout are likely more functionally important. We query the IMPC Solr API to count the number of significant phenotypes for each gene in our analysis.

**Note:** IMPC queries use gene lists and can be run independently of other D-series notebooks.

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

DATA_DIR = Path('data/external')
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMPC_OUT = DATA_DIR / 'impc_phenotype_counts.csv'

GENE_API = 'https://www.ebi.ac.uk/mi/impc/solr/gene/select'
STAT_API = 'https://www.ebi.ac.uk/mi/impc/solr/statistical-result/select'

if IMPC_OUT.exists():
    impc = pd.read_csv(IMPC_OUT)
    print(f'IMPC phenotype counts already cached: {len(impc)} genes')
else:
    # Load gene symbols from Geneformer token dictionary (cloned by D01)
    import pickle
    name_id_path = Path('Geneformer/geneformer/gene_name_id_dict_gc104M.pkl')
    if name_id_path.exists():
        with open(name_id_path, 'rb') as _f:
            name_id_dict = pickle.load(_f)
        gene_symbols = list(name_id_dict.keys())
    else:
        raise FileNotFoundError(
            'Geneformer token dictionary not found — run D01 first')
    
    print(f'Querying IMPC for {len(gene_symbols):,} genes...')
    
    # Step 1: Map human symbols to MGI accession IDs
    batch_size = 50
    gene_to_mgi = {}
    
    for i in range(0, len(gene_symbols), batch_size):
        batch = gene_symbols[i:i + batch_size]
        query = ' OR '.join(f'"{g}"' for g in batch)
        params = {
            'q': f'human_gene_symbol:({query})',
            'rows': 200,
            'fl': 'human_gene_symbol,mgi_accession_id',
            'wt': 'json'
        }
        resp = requests.get(GENE_API, params=params)
        resp.raise_for_status()
        docs = resp.json()['response']['docs']
        for doc in docs:
            sym = doc.get('human_gene_symbol')
            mgi = doc.get('mgi_accession_id')
            if sym and mgi:
                # sym can be a list
                if isinstance(sym, list):
                    for s in sym:
                        gene_to_mgi[s] = mgi
                else:
                    gene_to_mgi[sym] = mgi
        
        if (i // batch_size) % 10 == 0:
            print(f'  Gene lookup: {min(i + batch_size, len(gene_symbols)):,} / {len(gene_symbols):,}')
        time.sleep(0.5)  # Be polite to the API
    
    print(f'  Mapped {len(gene_to_mgi):,} genes to MGI IDs')
    
    # Step 2: Count significant phenotypes per MGI ID
    records = []
    mgi_ids = list(set(gene_to_mgi.values()))
    
    for i in range(0, len(mgi_ids), batch_size):
        batch = mgi_ids[i:i + batch_size]
        query = ' OR '.join(f'"{m}"' for m in batch)
        params = {
            'q': f'marker_accession_id:({query})',
            'rows': 0,
            'facet': 'true',
            'facet.field': 'marker_accession_id',
            'facet.limit': len(batch),
            'fq': 'significant:true',
            'wt': 'json'
        }
        resp = requests.get(STAT_API, params=params)
        resp.raise_for_status()
        facets = resp.json()['facet_counts']['facet_fields']['marker_accession_id']
        
        # Facet response is alternating [id, count, id, count, ...]
        for j in range(0, len(facets), 2):
            mgi_id = facets[j]
            count = facets[j + 1]
            records.append({'mgi_accession_id': mgi_id, 'n_phenotypes': count})
        
        if (i // batch_size) % 10 == 0:
            print(f'  Phenotype counts: {min(i + batch_size, len(mgi_ids)):,} / {len(mgi_ids):,}')
        time.sleep(0.5)
    
    # Merge back to gene symbols
    mgi_to_genes = {}
    for sym, mgi in gene_to_mgi.items():
        mgi_to_genes.setdefault(mgi, []).append(sym)
    
    pheno_df = pd.DataFrame(records)
    result_rows = []
    for _, row in pheno_df.iterrows():
        syms = mgi_to_genes.get(row['mgi_accession_id'], [])
        for sym in syms:
            result_rows.append({
                'gene': sym,
                'mgi_accession_id': row['mgi_accession_id'],
                'n_phenotypes': row['n_phenotypes']
            })
    
    impc = pd.DataFrame(result_rows)
    impc = impc.drop_duplicates(subset=['gene']).reset_index(drop=True)
    impc.to_csv(IMPC_OUT, index=False)
    print(f'  Saved {len(impc)} genes to {IMPC_OUT}')

IMPC phenotype counts already cached: 8119 genes


## Section 2: Replogle Perturb-seq Data

Replogle et al. (2022) performed genome-scale Perturb-seq in K562 cells. Their supplementary Table A provides per-gene summary statistics including the mean leverage score (a measure of how much each genetic perturbation affects the transcriptomic landscape). These tables must be manually downloaded from the publication.

**Download instructions:**
1. Go to https://doi.org/10.1016/j.cell.2022.05.013 → Supplemental Information
2. Download Table S2 (Excel file: ~50 MB)
3. Open in Excel. The relevant sheets are: Tab A (K562 day 8 summary statistics), Tab B (K562 day 6 summary statistics)
4. Export each sheet as CSV to data/external/1-s2/ with filenames matching: TabA_K562_day8_summary_stat-Table 1.csv, TabB_K562_day6_summary_stat-Table 1.csv

**Reference:** Replogle JM et al. "Mapping information-rich genotype-phenotype landscapes with genome-scale Perturb-seq." Cell 185, 2559–2575 (2022). https://doi.org/10.1016/j.cell.2022.05.013

In [2]:
REPLOGLE_DIR = DATA_DIR / '1-s2'
REPLOGLE_DIR.mkdir(parents=True, exist_ok=True)

TAB_A = REPLOGLE_DIR / 'TabA_K562_day8_summary_stat-Table 1.csv'
TAB_B = REPLOGLE_DIR / 'TabB_K562_day6_summary_stat-Table 1.csv'

print('Replogle perturb-seq supplementary tables')
print('=' * 50)
print()
print('These files must be downloaded manually from the publication:')
print('  Replogle JM et al., Cell 185, 2559–2575 (2022)')
print('  https://doi.org/10.1016/j.cell.2022.05.013')
print()
print('Download Table S2 (Excel) from the Cell website supplementary materials,')
print('then export the relevant sheets as CSV:')
print(f'  → Tab A (K562 day 8): {TAB_A}')
print(f'  → Tab B (K562 day 6): {TAB_B}')
print()

for name, path in [('Tab A (K562 day 8)', TAB_A), ('Tab B (K562 day 6)', TAB_B)]:
    if path.exists():
        df = pd.read_csv(path)
        print(f'  ✓ {name}: {len(df):,} rows, {len(df.columns)} columns')
    else:
        print(f'  ✗ {name}: NOT FOUND at {path}')

Replogle perturb-seq supplementary tables

These files must be downloaded manually from the publication:
  Replogle JM et al., Cell 185, 2559–2575 (2022)
  https://doi.org/10.1016/j.cell.2022.05.013

Download Table S2 (Excel) from the Cell website supplementary materials,
then export the relevant sheets as CSV:
  → Tab A (K562 day 8): data/external/1-s2/TabA_K562_day8_summary_stat-Table 1.csv
  → Tab B (K562 day 6): data/external/1-s2/TabB_K562_day6_summary_stat-Table 1.csv

  ✓ Tab A (K562 day 8): 11,258 rows, 17 columns
  ✓ Tab B (K562 day 6): 2,285 rows, 17 columns


## Section 3: Verification Summary

In [3]:
print('External validation datasets — Status Summary')
print('=' * 50)
print()

outputs = [
    ('IMPC phenotype counts', IMPC_OUT),
    ('Replogle Tab A (K562 day 8)', TAB_A),
    ('Replogle Tab B (K562 day 6)', TAB_B),
]

for name, path in outputs:
    if path.exists():
        size = path.stat().st_size / 1024  # KB
        print(f'  ✓ {name}')
        print(f'    → {path}')
        print(f'    → Size: {size:.1f} KB')
    else:
        print(f'  ✗ {name}')
        print(f'    → {path} (NOT FOUND)')
    print()

print('Ready for external validation analysis.')

External validation datasets — Status Summary

  ✓ IMPC phenotype counts
    → data/external/impc_phenotype_counts.csv
    → Size: 160.7 KB

  ✓ Replogle Tab A (K562 day 8)
    → data/external/1-s2/TabA_K562_day8_summary_stat-Table 1.csv
    → Size: 2493.8 KB

  ✓ Replogle Tab B (K562 day 6)
    → data/external/1-s2/TabB_K562_day6_summary_stat-Table 1.csv
    → Size: 504.1 KB

Ready for external validation analysis.
